# Challenge 02 - Cats vs Dogs (HOG features)

## Data Preparation and Loading

The dataset contains 900 labeled RGB images of cats (`0`) and dogs (`1`), plus an unlabeled test set. The images are loaded with the helper functions of the course module `ml_utilities`, shuffled with a fixed seed, and kept in their original size so that they can be resized at different resolutions later. No feature other than HOG is used, no external images are added and no data augmentation is applied. Image preprocessing is identical for training and test images and does not change the number of patterns.

In [1]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

import time
import numpy as np
from joblib import Memory
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from skimage import color, exposure, filters, feature

import utilities

# Configuration and Data Loading
db_path = '../DBs/CaniGatti_ML18'
exp_path = 'Experiments'
train_filelist = 'BinaryTrainingSet.txt'
test_filelist = 'Unlabeled_BinaryTestSet.txt'

result_path_solution1 = '../pred_sol1.txt'
result_path_solution2 = '../pred_sol2.txt'
result_path_solution3 = '../pred_sol3.txt'
result_path_final = '../Es4Predictions.txt'

# HOG and pipeline parameters
image_side = 128
orientations = 12
cell_side = 16
alpha = 0.5
n_pca = 256

# Disk cache: re-running a function with the same parameters is instantaneous
memory = Memory(exp_path, verbose=0)

start = time.time()
train_orig_x, train_y = utilities.load_labeled_dataset(train_filelist, db_path, cache=memory)
test_orig_x = utilities.load_unlabeled_dataset(test_filelist, db_path, cache=memory)
utilities.shuffle_in_unison([train_orig_x, train_y], seed=1234)

print('Loaded %d training and %d test images in %.2f s.' % (len(train_orig_x), len(test_orig_x), time.time() - start))
print('Cats:', np.count_nonzero(train_y == 0))
print('Dogs:', np.count_nonzero(train_y == 1))

Loaded 900 training and 150 test images in 0.20 s.
Cats: 509
Dogs: 391


In [2]:
def save_predictions(predictions, path):
    with open(path, 'w') as f:
        for prediction in predictions:
            f.write(f'{int(prediction)}\n')
    print(f'Predictions successfully saved to {path}')

In [3]:
def extract_hog_features(images):
    """Resize to image_side x image_side, convert to grayscale, and extract HOG features."""
    resized = utilities.resize_images(images, image_side, image_side, cache=memory)
    hog_features = []
    for img in resized:
        hog = feature.hog(color.rgb2gray(img),
                          orientations=orientations,
                          pixels_per_cell=(cell_side, cell_side),
                          cells_per_block=(2, 2),
                          block_norm='L2-Hys')
        hog_features.append(hog)
    return np.array(hog_features)

In [4]:
def power_transform(X, alpha=0.5):
    """Element-wise x**alpha (HOG L2-Hys is >= 0; clipping guards against -0.0)."""
    return np.power(np.clip(X, 0, None), alpha)

## Solution 1

This solution extracts 128×128 px HOG features (12 orientations, 16×16 cells) and processes them through a pipeline with an element-wise square-root transform (`alpha=0.5`), standard scaling, and PCA reduction to 256 components. Classification is performed via an RBF-kernel SVC (`C=3`, `gamma=6e-4`). The entire pipeline is evaluated through repeated stratified cross-validation before training on the full training set to produce test predictions.

In [5]:
def center_crop(img, frac=0.8):
    h, w = img.shape[:2]
    dh, dw = int(h * (1 - frac) / 2), int(w * (1 - frac) / 2)
    return img[dh:h - dh, dw:w - dw]

def extract_variant(images, cfg, variant):
    size = cfg['size']
    if variant == 'crop80':
        images = [center_crop(im) for im in images]
    resized = utilities.resize_images(images, size, size, cache=memory)
    hog_kw = dict(orientations=cfg['orientations'],
                  pixels_per_cell=(cfg['cell'], cfg['cell']),
                  cells_per_block=(2, 2), block_norm='L2-Hys')
    feats = []
    for img in resized:
        if variant == 'color':
            f = feature.hog(img, channel_axis=-1, **hog_kw)
        else:
            g = color.rgb2gray(img)
            if variant == 'equalize':
                g = exposure.equalize_hist(g)
            elif variant == 'blur':
                g = filters.gaussian(g, sigma=1)
            f = feature.hog(g, transform_sqrt=(variant == 'hog_sqrt'), **hog_kw)
        feats.append(f)
    return np.array(feats)

In [6]:
# Preprocessing and HOG extraction (training)
train_feature_x = extract_hog_features(train_orig_x)
print('HOG train shape:', train_feature_x.shape)

# Build the classifier
clf = Pipeline([
    ('pow', FunctionTransformer(power_transform, kw_args={'alpha': alpha})),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=n_pca, random_state=1234)),
    ('svc', SVC(kernel='rbf', C=3, gamma=6e-4)),
])

# Preprocessing and HOG extraction (test), identical to training
test_feature_x = extract_hog_features(test_orig_x)
print('HOG test shape:', test_feature_x.shape)
assert test_feature_x.shape[1] == train_feature_x.shape[1]

# Evaluate the pipeline with repeated stratified cross-validation
cv_solution1 = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=2024)
scores_solution1 = cross_val_score(clf, train_feature_x, train_y, cv=cv_solution1, scoring='accuracy', n_jobs=4)
print('CV accuracy: %.4f +/- %.4f' % (scores_solution1.mean(), scores_solution1.std()))

HOG train shape: (900, 2352)
HOG test shape: (150, 2352)
CV accuracy: 0.8131 +/- 0.0279


### Solution 1 Prediction and Export

The test images are processed with the best HOG configuration found above. The predictions are saved in `pred_sol1.txt`.

In [7]:
# Train on the whole training set and save predictions
clf.fit(train_feature_x, train_y)
predictions_solution1 = clf.predict(test_feature_x)

print('Predictions: %d cats / %d dogs' % (np.count_nonzero(predictions_solution1 == 0),
                                          np.count_nonzero(predictions_solution1 == 1)))
save_predictions(predictions_solution1, result_path_solution1)

Predictions: 89 cats / 61 dogs
Predictions successfully saved to ../pred_sol1.txt


## Solution 2

In [8]:
# TODO

### Solution 2 Prediction and Export

The ensemble is fitted on the whole training set and its predictions are saved in `pred_sol2.txt`.

In [9]:
# TODO

### Solution 3

In [10]:
# TODO

### Solution 3 Prediction and Export

The ensemble is fitted on the whole training set and its predictions are saved in `pred_sol3.txt`.

In [11]:
# TODO